# KIWA NEXRAD analysis — ASU West Valley ARM site
### 18–19 November 2025 afternoon supercell / hail event

This notebook reproduces, end-to-end:

1. **Data ingest** — NEXRAD Level II volume scans for radar **KIWA** (Phoenix / Williams Gateway
   WSR-88D) pulled from the Google Cloud public bucket `gcp-public-data-nexrad-l2`
   (a fallback for the AWS NEXRAD archive).
2. **Rain-rate time series** over the ARM site footprint using a Marshall–Palmer *Z–R* relation
   (`Z = 200 R^1.6`), with a **hail cap** at 53 dBZ to bound hail contamination.
3. **A 15-frame animation** of 0.5° reflectivity with a tiled OpenStreetMap basemap, interstates,
   the Phoenix airport marker, and a synchronised rain-rate cursor — assembled into a GIF.

**Site:** ASU West Valley ARM site, 33.606726° N, 112.166912° W.
**Colormap:** `ChaseSpectral` from **cmweather** (0–70 dBZ).

**Georeferencing note (important):** the reflectivity field is plotted using Py-ART's own
`gate_longitude` / `gate_latitude` on a **Mercator** GeoAxes via `pcolormesh(..., transform=PlateCarree())`.
We deliberately avoid a cartopy `AzimuthalEquidistant` axes centred on the radar — its transform is
inaccurate at the origin for a non-zero central latitude and shifts the gate field ~21 km south of the
radar marker.

---
## 0. Environment

Requires: `arm-pyart`, `cmweather`, `cartopy`, `matplotlib`, `numpy`, `pandas`, `pillow`.
Network access to the GCS bucket host and (for the basemap) the OSM tile servers is needed for a
fresh run.

In [ ]:
import os, math, glob, urllib.request, tarfile
import numpy as np, numpy.ma as ma, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
import pyart
import cmweather  # registers the cmweather colormaps (ChaseSpectral, etc.)
from PIL import Image

print("Py-ART", pyart.__version__)

---
## 1. Helper functions

These are the correctly-georeferenced Py-ART helpers (also published as the `nexrad-radar-gcs`
skill). They are inlined here so the notebook is fully self-contained.

- `nexrad_gcs_keys` / `download_nexrad` — list & fetch Level II volume keys from the GCS bucket.
- `read_radar_safe` — read a scan, returning `None` for the occasional corrupt/truncated volume.
- `dualpol_gatefilter` — standard dual-pol met filter (ρHV ≥ 0.85, dBZ ≥ 5, drop invalid).
- `make_radar_geoaxes` / `add_context_features` / `plot_ppi_map` — the Mercator PPI plotting stack.
- `setup_cmweather` — registers colormaps and resets `savefig.bbox` (which `tight` would collapse on a GeoAxes).

In [ ]:
import math

# --- Google Cloud public NEXRAD Level-II bucket (AWS fallback) ---
NEXRAD_GCS_HOST = "https://gcp-public-data-nexrad-l2.storage.googleapis.com"
# Marshall-Palmer Z-R relation: Z = A * R**B  (Z in linear mm^6 m^-3)
MARSHALL_PALMER_A = 200.0
MARSHALL_PALMER_B = 1.6


def scan_time(path):
    """Parse the UTC datetime from a NEXRAD key/filename (SITEyyyymmdd_HHMMSS...)."""
    import re, datetime as dt
    m = re.search(r"[A-Z]{4}(\d{8})_(\d{6})", str(path))
    if not m:
        raise ValueError("no NEXRAD timestamp in %r" % path)
    return dt.datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S")


def nexrad_gcs_keys(site, year, month, day, host=None, full_volumes_only=True):
    """List Level-II volume-scan object keys for one radar SITE on one UTC day
    from the Google Cloud public NEXRAD bucket. Returns a sorted list of keys.
    Requires network access to the bucket host (request_network_access)."""
    import urllib.request, xml.etree.ElementTree as ET
    if host is None:
        host = NEXRAD_GCS_HOST
    prefix = "%04d/%02d/%02d/%s/" % (int(year), int(month), int(day), site)
    keys, marker = [], ""
    while True:
        url = "%s/?prefix=%s&marker=%s" % (host, prefix, marker)
        with urllib.request.urlopen(url, timeout=60) as r:
            root = ET.fromstring(r.read())
        contents = root.findall("{*}Contents")
        if not contents:
            break
        for c in contents:
            keys.append(c.find("{*}Key").text)
        trunc = root.find("{*}IsTruncated")
        if trunc is None or (trunc.text or "").lower() != "true":
            break
        marker = keys[-1]
    if full_volumes_only:  # drop supplemental *_MDM message volumes
        keys = [k for k in keys if not k.endswith("_MDM")]
    return sorted(keys)


def download_nexrad(keys, dest_dir="nexrad_scans", host=None):
    """Download NEXRAD keys from the GCS bucket into dest_dir (skips existing).
    Returns local file paths sorted by scan time."""
    import os, urllib.request
    if host is None:
        host = NEXRAD_GCS_HOST
    os.makedirs(dest_dir, exist_ok=True)
    paths = []
    for k in keys:
        local = os.path.join(dest_dir, os.path.basename(k))
        if not os.path.exists(local):
            urllib.request.urlretrieve("%s/%s" % (host, k), local)
        paths.append(local)
    return sorted(paths, key=scan_time)


def read_radar_safe(path):
    """Read a NEXRAD archive with Py-ART; return None on a corrupt/unreadable file
    (some volume scans in the archive are truncated)."""
    import pyart
    try:
        return pyart.io.read_nexrad_archive(path)
    except Exception:
        return None


def setup_cmweather():
    """Register cmweather colormaps and fix the savefig bbox. Call once before plotting.
    figure-style's apply_figure_style() sets savefig.bbox='tight', which COLLAPSES a
    cartopy GeoAxes on save -- this resets it to 'standard'."""
    import cmweather  # noqa: F401  (import registers the colormaps)
    import matplotlib as mpl
    mpl.rcParams["savefig.bbox"] = "standard"
    return True


def dualpol_gatefilter(radar, rhohv_min=0.85, dbz_min=5.0, field="reflectivity"):
    """Standard dual-pol meteorological gate filter: drop low rho_HV, low dBZ, invalid."""
    import pyart
    gf = pyart.filters.GateFilter(radar)
    gf.exclude_below("cross_correlation_ratio", rhohv_min)
    gf.exclude_below(field, dbz_min)
    gf.exclude_invalid(field)
    return gf


def extract_site_rainrate(radar, site_lat, site_lon, radius_km=2.0,
                          rhohv_min=0.85, dbz_min=5.0, zr_a=None, zr_b=None):
    """Mean reflectivity and Marshall-Palmer rain rate over a circular footprint
    around a site, from the lowest sweep. Averages in LINEAR Z then converts.
    Returns dict(mean_dbz, rain_rate_mmph, n_gates)."""
    import numpy as np, numpy.ma as ma
    if zr_a is None:
        zr_a = MARSHALL_PALMER_A
    if zr_b is None:
        zr_b = MARSHALL_PALMER_B
    sweep0 = radar.extract_sweeps([0])
    lats = sweep0.gate_latitude["data"]
    lons = sweep0.gate_longitude["data"]
    dy = (lats - site_lat) * 111.0
    dx = (lons - site_lon) * 111.0 * math.cos(math.radians(site_lat))
    near = np.sqrt(dx ** 2 + dy ** 2) <= radius_km
    dbz = sweep0.fields["reflectivity"]["data"]
    rho = sweep0.fields["cross_correlation_ratio"]["data"]
    good = (near & ~ma.getmaskarray(dbz) & (dbz >= dbz_min)
            & ~ma.getmaskarray(rho) & (rho >= rhohv_min))
    n = int(np.count_nonzero(good))
    if n == 0:
        return {"mean_dbz": float("nan"), "rain_rate_mmph": float("nan"), "n_gates": 0}
    zlin = 10.0 ** (np.asarray(dbz[good]) / 10.0)
    zmean = float(np.mean(zlin))
    return {"mean_dbz": 10.0 * math.log10(zmean),
            "rain_rate_mmph": float((zmean / zr_a) ** (1.0 / zr_b)),
            "n_gates": n}


def make_radar_geoaxes(fig, radar, rect=(0.05, 0.05, 0.9, 0.9), extent=None,
                       tiler=None, tile_zoom=9, tile_alpha=0.5):
    """Create a correctly-georeferenced GeoAxes for a radar PPI.

    Uses a MERCATOR projection. IMPORTANT georeferencing lesson: do NOT use a
    cartopy AzimuthalEquidistant centered on the radar. cartopy's aeqd transform
    is inaccurate at the origin for a non-zero central_latitude -- it maps the
    radar's own lon/lat to roughly (0, +21 km) instead of (0, 0), so the gate
    field (drawn on that projection) ends up shifted ~21 km south of the radar
    marker (which is plotted from true lon/lat). The symptom is a radar triangle
    sitting NORTH of the cone-of-silence blind zone. Mercator is well-behaved
    over a metro-scale extent, and we plot the data using Py-ART's OWN
    gate_longitude/gate_latitude (spherical, correct) via pcolormesh with a
    PlateCarree transform -- so no aeqd round-trip happens anywhere.

    rect   = [left, bottom, width, height] in figure fraction.
    extent = [lon_min, lon_max, lat_min, lat_max] (lon/lat); optional.
    tiler  = a cartopy.io.img_tiles source (e.g. cimgt.OSM(cache=True)) to draw a
             tiled basemap under the radar; requires network access to the tile
             host (for OSM: a/b/c.tile.openstreetmap.org). None = no basemap.
    Returns the GeoAxes. Draw data/markers with transform=ccrs.PlateCarree().
    """
    import cartopy.crs as ccrs
    proj = tiler.crs if tiler is not None else ccrs.Mercator()
    ax = fig.add_axes(list(rect), projection=proj)
    if extent is not None:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    if tiler is not None:
        try:
            ax.add_image(tiler, tile_zoom, alpha=tile_alpha)
        except Exception:
            pass  # tiles blocked/unavailable -- fall back to a bare basemap
    return ax


def add_context_features(ax, extent, roads=True, airports=True, states=True,
                         label_interstates=True):
    """Overlay interstates, airports, and state borders on a radar GeoAxes.

    extent = [lon_min, lon_max, lat_min, lat_max]. Interstates (level=='Interstate')
    are drawn in red with route labels; beltways/expressways in orange; airports
    as navy diamonds with IATA/abbrev labels. All from Natural Earth 10m cultural
    layers (allowlisted CDN). Draw AFTER the radar pcolormesh so lines sit on top.
    """
    import cartopy.crs as ccrs, cartopy.feature as cfeature
    import cartopy.io.shapereader as shpreader
    from shapely.geometry import box
    pc = ccrs.PlateCarree()
    clip = box(extent[0], extent[2], extent[1], extent[3])
    if states:
        ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="0.4", linewidth=0.6)
    if roads:
        rd = shpreader.Reader(shpreader.natural_earth(resolution="10m",
                              category="cultural", name="roads"))
        labeled = set()
        for rec in rd.records():
            g = rec.geometry
            if g is None or not g.intersects(clip):
                continue
            lvl = rec.attributes.get("level"); typ = rec.attributes.get("type")
            nm = str(rec.attributes.get("name", "")); gi = g.intersection(clip)
            if lvl == "Interstate":
                ax.add_geometries([gi], crs=pc, edgecolor="#b8231f",
                                  facecolor="none", linewidth=2.2, zorder=6)
                if label_interstates and nm and ("I", nm) not in labeled:
                    pt = gi.representative_point()
                    ax.text(pt.x, pt.y, "I-" + nm, fontsize=8, color="#b8231f",
                            fontweight="bold", transform=pc, zorder=9, ha="center",
                            bbox=dict(boxstyle="round,pad=0.1", fc="white",
                                      ec="none", alpha=0.7))
                    labeled.add(("I", nm))
            elif typ == "Beltway" or rec.attributes.get("expressway", 0) == 1:
                ax.add_geometries([gi], crs=pc, edgecolor="#e08214",
                                  facecolor="none", linewidth=1.3, zorder=6)
    if airports:
        ap = shpreader.Reader(shpreader.natural_earth(resolution="10m",
                              category="cultural", name="airports"))
        for rec in ap.records():
            g = rec.geometry
            if g is None or not g.within(clip):
                continue
            ax.plot(g.x, g.y, marker="D", color="#08306b", markersize=6,
                    transform=pc, zorder=8)
            nm = (rec.attributes.get("iata_code") or rec.attributes.get("abbrev")
                  or rec.attributes.get("name", ""))
            ax.annotate(str(nm), xy=(g.x, g.y), xytext=(4, 3),
                        textcoords="offset points", fontsize=7, color="#08306b",
                        fontweight="bold", transform=pc, zorder=8)


def plot_ppi_map(radar, ax, fig, field="reflectivity", sweep=0, extent=None,
                 cmap="ChaseSpectral", vmin=0.0, vmax=70.0, rhohv_min=0.85,
                 dbz_min=5.0, site=None, title="", colorbar_label=None,
                 alpha=0.85, add_colorbar=True, context=False):
    """Plot a dual-pol-filtered PPI field on a cartopy GeoAxes `ax`.

    extent = [lon_min, lon_max, lat_min, lat_max]; site = (lat, lon).
    Call setup_cmweather() once before the first plot.

    GEOREFERENCING (verified correct): the field is drawn with Py-ART's OWN
    radar.gate_longitude / radar.gate_latitude (spherical geometry, matches the
    true radar location to ~6 decimals) via ax.pcolormesh(..., transform=
    ccrs.PlateCarree()). This deliberately does NOT go through
    RadarMapDisplay.plot_ppi_map on a cartopy AzimuthalEquidistant axes: that
    path shifts the gate field ~21 km south of the radar marker because cartopy's
    aeqd transform is inaccurate at the origin for non-zero central_latitude.
    Build `ax` with make_radar_geoaxes(fig, radar, ...) (Mercator). Markers are
    drawn with transform=ccrs.PlateCarree() (lon/lat).

    context=True overlays interstates + airports + state borders via
    add_context_features(). Returns the QuadMesh from pcolormesh.
    """
    import numpy as np
    import cartopy.crs as ccrs
    pc = ccrs.PlateCarree()
    gf = dualpol_gatefilter(radar, rhohv_min, dbz_min, field)
    sl = radar.get_slice(sweep)
    data = radar.get_field(sweep, field)
    excl = gf.gate_excluded[sl]
    data = np.ma.masked_where(excl, data)
    glon = radar.gate_longitude["data"][sl]
    glat = radar.gate_latitude["data"][sl]
    pm = ax.pcolormesh(glon, glat, data, transform=pc, cmap=cmap,
                       vmin=vmin, vmax=vmax, alpha=alpha, shading="auto", zorder=5)
    if extent is not None:
        ax.set_extent(extent, crs=pc)
    if context and extent is not None:
        add_context_features(ax, extent)
    if add_colorbar:
        cb = fig.colorbar(pm, ax=ax, shrink=0.8, pad=0.02)
        cb.set_label(colorbar_label or field)
    if title:
        ax.set_title(title)
    if site is not None:
        ax.plot(site[1], site[0], marker="*", markersize=17, color="black",
                markerfacecolor="yellow", markeredgewidth=1.2, transform=pc, zorder=11)
    ax.plot(radar.longitude["data"][0], radar.latitude["data"][0],
            marker="^", markersize=10, color="black", markerfacecolor="white",
            markeredgewidth=1.2, transform=pc, zorder=11)
    return pm

---
## 2. Configuration

In [ ]:
SITE_ID   = "KIWA"                       # Phoenix / Williams Gateway WSR-88D
YEAR, MONTH, DAYS = 2025, 11, (18, 19)   # UTC days spanning the event
site_lat, site_lon = 33.606726, -112.166912   # ASU West Valley ARM site
site = (site_lat, site_lon)

# Map extent [lon_min, lon_max, lat_min, lat_max] and reflectivity colour scale
extent = [-112.7, -111.0, 32.6, 34.0]
CMAP, VMIN, VMAX = "ChaseSpectral", 0, 70

# Z-R (Marshall-Palmer) and gate-filter parameters
ZR_A, ZR_B  = 200.0, 1.6
RHOHV_MIN, DBZ_MIN = 0.85, 5.0
HAIL_CAP    = 53.0                       # dBZ ceiling before linear averaging

setup_cmweather()

---
## 3. Download NEXRAD Level II volumes from Google Cloud

Volume scans are stored as per-scan tar archives on the GCS bucket. We list the keys for each UTC
day, download the tarballs, and extract the individual `.ar2v` scan files. Existing files are
skipped, so re-running is cheap.

In [ ]:
os.makedirs("nov_tars", exist_ok=True)
os.makedirs("nov_scans", exist_ok=True)

allkeys = []
for day in DAYS:
    kk = nexrad_gcs_keys(SITE_ID, YEAR, MONTH, day)
    allkeys += kk
    print(f"{YEAR}-{MONTH:02d}-{day:02d}: {len(kk)} volumes")

host_url = "https://gcp-public-data-nexrad-l2.storage.googleapis.com/"
for i, k in enumerate(allkeys):
    tarpath = os.path.join("nov_tars", os.path.basename(k))
    if not os.path.exists(tarpath):
        urllib.request.urlretrieve(host_url + k, tarpath)
    with tarfile.open(tarpath) as tf:
        for m in (m for m in tf.getmembers() if m.isfile()):
            outp = os.path.join("nov_scans", os.path.basename(m.name))
            if not os.path.exists(outp):
                with tf.extractfile(m) as src, open(outp, "wb") as dst:
                    dst.write(src.read())
    if (i + 1) % 20 == 0:
        print(f"  [{i+1}/{len(allkeys)}] downloaded/extracted", flush=True)

allv = sorted(glob.glob("nov_scans/*.ar2v"), key=scan_time)
print("total scan files:", len(allv))

---
## 4. Rain-rate time series over the ARM site

For every volume scan we compute the mean reflectivity within a footprint around the site
(2 km and 10 km radii), average in **linear Z**, and convert to rain rate with the Marshall–Palmer
relation. A **hail cap** clips gate reflectivity at 53 dBZ *before* averaging: reflectivities above
this are almost certainly hail contamination, and the uncapped Z–R rate would badly over-estimate
rainfall.

`extract_site_capped` returns both the raw and hail-capped rain rate plus the max gate dBZ in the
footprint.

In [ ]:
def extract_site_capped(radar, site_lat, site_lon, radius_km=2.0, rhohv_min=RHOHV_MIN,
                        dbz_min=DBZ_MIN, hail_cap=HAIL_CAP, zr_a=ZR_A, zr_b=ZR_B):
    """Site-footprint rain rate with a hail cap: gate dBZ is clipped at hail_cap
    before linear averaging + Z-R conversion. Returns capped + uncapped rain rate."""
    sweep0 = radar.extract_sweeps([0])
    lats = sweep0.gate_latitude["data"]; lons = sweep0.gate_longitude["data"]
    dy = (lats - site_lat) * 111.0
    dx = (lons - site_lon) * 111.0 * math.cos(math.radians(site_lat))
    near = np.sqrt(dx**2 + dy**2) <= radius_km
    dbz = sweep0.fields["reflectivity"]["data"]
    rho = sweep0.fields["cross_correlation_ratio"]["data"]
    good = (near & ~ma.getmaskarray(dbz) & (dbz >= dbz_min)
            & ~ma.getmaskarray(rho) & (rho >= rhohv_min))
    n = int(np.count_nonzero(good))
    if n == 0:
        return dict(rr=np.nan, rr_capped=np.nan, dbz_max=np.nan, n=0)
    g = np.asarray(dbz[good])
    zlin     = 10.0 ** (g / 10.0)
    zlin_cap = 10.0 ** (np.minimum(g, hail_cap) / 10.0)
    rr     = float((np.mean(zlin)     / zr_a) ** (1.0 / zr_b))
    rr_cap = float((np.mean(zlin_cap) / zr_a) ** (1.0 / zr_b))
    return dict(rr=rr, rr_capped=rr_cap, dbz_max=float(g.max()), n=n)

In [ ]:
out = []
for i, p in enumerate(allv):
    radar = read_radar_safe(p)
    if radar is None:          # skip corrupt/truncated volumes
        continue
    c2  = extract_site_capped(radar, site_lat, site_lon, 2.0)
    c10 = extract_site_capped(radar, site_lat, site_lon, 10.0)
    out.append(dict(time_utc=scan_time(p),
                    rr2_raw=c2["rr"],  rr2_cap=c2["rr_capped"],  dbzmax2=c2["dbz_max"],  n2=c2["n"],
                    rr10_raw=c10["rr"], rr10_cap=c10["rr_capped"], dbzmax10=c10["dbz_max"], n10=c10["n"]))
    del radar
    if (i + 1) % 40 == 0:
        print(f"{i+1}/{len(allv)}", flush=True)

df = pd.DataFrame(out).sort_values("time_utc").reset_index(drop=True)
df["tt"] = pd.to_datetime(df["time_utc"])
df.to_csv("nov1819_arm_site_timeseries.csv", index=False)

print("rows:", len(df))
print("peak 2 km  capped:", round(df.rr2_cap.max(), 2), "mm/h at",
      df.loc[df.rr2_cap.idxmax(), "time_utc"])
print("peak 10 km capped:", round(df.rr10_cap.max(), 2), "mm/h at",
      df.loc[df.rr10_cap.idxmax(), "time_utc"])

### 4a. Time-series figure

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.fill_between(df.tt, df.rr2_cap, color="#4C72B0", alpha=0.35)
ax.plot(df.tt, df.rr2_cap,  color="#4C72B0", lw=1.6, label="2 km (hail-capped)")
ax.plot(df.tt, df.rr10_cap, color="#2a9d5c", lw=1.6, label="10 km (hail-capped)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Rain rate over ARM site (mm h$^{-1}$)")
ax.set_title("Marshall\u2013Palmer rain rate (hail-capped at 53 dBZ)")
ax.legend(loc="upper right", frameon=False)
fig.tight_layout()
fig.savefig("nov1819_timeseries.png", dpi=150)
plt.show()

---
## 5. Afternoon animation (15 frames) + rain-rate cursor

We render the afternoon supercell window (≈20:03 → 21:38 UTC on 18 Nov). Each frame is a two-panel
figure:

- **Left:** KIWA 0.5° reflectivity on a Mercator GeoAxes with an OpenStreetMap basemap, interstates
  (I-17 / I-10 / I-8), the Phoenix airport marker, the radar (▲) and the ARM site (★).
- **Right:** the rain-rate time series with a red cursor at the current scan time.

The window is selected from the time-series index; one corrupt scan (~20:57 UTC) is skipped
automatically by `read_radar_safe`.

In [ ]:
# Select the afternoon window on 18 Nov and the matching scan files
win_lo = pd.Timestamp("2025-11-18 20:00")
win_hi = pd.Timestamp("2025-11-18 21:40")
xlo, xhi = pd.Timestamp("2025-11-18 19:30"), pd.Timestamp("2025-11-18 22:30")

path_time = {p: pd.to_datetime(scan_time(p)) for p in allv}
anim_vols = [p for p in allv if win_lo <= path_time[p] <= win_hi]
print(f"{len(anim_vols)} volumes in animation window")

# Shared OSM tiler (cached) reused across frames
_tiler = cimgt.OSM(cache=True)
os.makedirs("nov_frames", exist_ok=True)

In [ ]:
def render_frame(path, idx):
    """Two-panel frame: georeferenced PPI (left) + rain-rate cursor (right)."""
    r_f = read_radar_safe(path)
    if r_f is None:
        return None
    st = pd.Timestamp(pyart.util.datetime_from_radar(r_f).strftime("%Y-%m-%d %H:%M:%S"))

    fig = plt.figure(figsize=(15, 7))
    # LEFT: reflectivity PPI with basemap + context features
    axL = make_radar_geoaxes(fig, r_f, rect=(0.02, 0.06, 0.50, 0.88),
                             extent=extent, tiler=_tiler, tile_zoom=9, tile_alpha=0.5)
    plot_ppi_map(r_f, axL, fig, field="reflectivity", sweep=0, extent=extent,
                 cmap=CMAP, vmin=VMIN, vmax=VMAX, site=site,
                 colorbar_label="Reflectivity (dBZ)",
                 title="KIWA 0.5$\\degree$ reflectivity\n" + f"{st:%Y-%m-%d %H:%M:%S} UTC",
                 context=True, add_colorbar=True)

    # RIGHT: rain-rate time series with a time cursor
    axR = fig.add_axes([0.60, 0.12, 0.36, 0.76])
    axR.fill_between(df.tt, df.rr2_cap, color="#4C72B0", alpha=0.35)
    axR.plot(df.tt, df.rr2_cap,  color="#4C72B0", lw=1.6, label="2 km (hail-capped)")
    axR.plot(df.tt, df.rr10_cap, color="#2a9d5c", lw=1.6, label="10 km (hail-capped)")
    axR.axvline(st, color="red", lw=2)
    j = (df.tt - st).abs().idxmin()
    axR.plot(df.tt[j], df.rr2_cap[j], "o", color="red", ms=9)
    axR.set_xlim(xlo, xhi); axR.set_ylim(0, 80)
    axR.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axR.set_xlabel("Time (UTC)")
    axR.set_ylabel("Rain rate over ARM site (mm h$^{-1}$)")
    axR.set_title("Marshall\u2013Palmer rain rate (hail-capped at 53 dBZ)")
    axR.legend(loc="upper right", frameon=False)

    p = f"nov_frames/f_{idx:02d}.png"
    fig.savefig(p, dpi=100)
    plt.close(fig)
    return p

In [ ]:
frame_paths = []
for idx, path in enumerate(anim_vols):
    p = render_frame(path, idx)
    if p is not None:
        frame_paths.append(p)
    print(idx, "ok" if p else "SKIP (corrupt)")
print("rendered", len(frame_paths), "frames")

### 5a. Assemble the GIF

No `ffmpeg` in the environment, so we build the animated GIF with Pillow.

In [ ]:
frame_paths = sorted(glob.glob("nov_frames/f_*.png"))
imgs = [Image.open(f).convert("RGB") for f in frame_paths]
out_gif = "nov1819_afternoon_animation.gif"
imgs[0].save(out_gif, save_all=True, append_images=imgs[1:], duration=450, loop=0)
print("wrote", out_gif, "-", os.path.getsize(out_gif), "bytes,", len(imgs), "frames")

---
## 6. Findings

- The afternoon supercell drives a hail core directly over the ARM site. The hail-capped 2 km rain
  rate **peaks at ~72.5 mm/h (63.5 dBZ) at 20:41 UTC** on 18 Nov — a hail-contaminated upper bound
  (the *uncapped* Z–R rate would read ~170 mm/h, which is not physical rainfall).
- A second, weaker cell clips the site near 21:08 UTC (~25 mm/h).
- The strongest *clean* (rain-only) over-site rate in the two-day window is a modest ~4.4 mm/h at
  ~09:49 UTC on 19 Nov.

**Method summary:** 0.5° sweep; dual-pol gate filter ρHV ≥ 0.85 and dBZ ≥ 5; footprint-mean in
linear Z; Marshall–Palmer `Z = 200 R^1.6`; hail cap at 53 dBZ.

**Outputs written:** `nov1819_arm_site_timeseries.csv`, `nov1819_timeseries.png`,
`nov1819_afternoon_animation.gif`, and per-frame PNGs in `nov_frames/`.